# PCA on Wine Data – With and Without PCA

Standardize the Wine dataset, inspect covariance structure, compute principal components, visualize explained variance, and compare a classifier with and without PCA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
print(X.shape)
X.head()

## 1. Standardization

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Approximate means:", np.round(X_scaled.mean(axis=0), 3))
print("Approximate variances:", np.round(X_scaled.var(axis=0), 3))

## 2. Covariance matrix

In [ ]:
cov_matrix = np.cov(X_scaled, rowvar=False)
cov_df = pd.DataFrame(cov_matrix, index=X.columns, columns=X.columns)
cov_df.round(3).head()

## 3. PCA and explained variance

In [ ]:
pca_full = PCA()
X_pca = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
print("Explained variance ratio:")
print(np.round(explained, 4))
print("Cumulative:")
print(np.round(np.cumsum(explained), 4))

plt.figure(figsize=(8,4))
plt.plot(range(1, len(explained)+1), np.cumsum(explained), marker="o")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.grid(True)
plt.show()

## 4. Visualize the first two PCs

In [ ]:
plt.figure(figsize=(7,5))
for cls in sorted(y.unique()):
    mask = y == cls
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f"Class {cls}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Wine data in PCA space")
plt.legend()
plt.show()

## 5. Compare classification with and without PCA

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

without_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

with_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("model", LogisticRegression(max_iter=3000))
])

without_pca.fit(X_train, y_train)
with_pca.fit(X_train, y_train)

pred_without = without_pca.predict(X_test)
pred_with = with_pca.predict(X_test)

print("Accuracy without PCA:", accuracy_score(y_test, pred_without))
print("Accuracy with 2 PCs:", accuracy_score(y_test, pred_with))

### Conclusion
PCA is unsupervised dimensionality reduction. It rotates the standardized feature space into orthogonal directions ordered by variance. Using fewer PCs can simplify a model, but it may reduce predictive information; therefore compare downstream performance.